# Retrieval-Augmented Generation (RAG) for Tesla Annual Reports

## Project Overview

This project demonstrates the implementation of a Retrieval-Augmented Generation (RAG) system for answering questions from Tesla's annual reports (2019–2023).

The solution combines:

- Vector embeddings using Sentence Transformers
- ChromaDB for vector storage and retrieval
- Llama 3.1 70B for answer generation
- Advanced retrieval using Hypothetical Question Indexing

The objective is to improve factual accuracy and reduce hallucinations by grounding responses in company filings.

---

## Business Use Case

Financial analysts often need to extract information from hundreds of pages of annual reports. Traditional keyword search is limited because users rarely phrase questions exactly as information appears in documents.

This RAG system enables:

- Natural language querying
- Semantic document retrieval
- Grounded financial analysis
- Explainable AI responses

---

## Architecture

User Question
↓
Embedding Model
↓
Vector Search (ChromaDB)
↓
Relevant Document Chunks
↓
LLM (Llama 3.1 70B)
↓
Final Answer


# Dataset Description

The dataset consists of Tesla Annual Reports (Form 10-K) spanning 2019–2023.

These reports contain:

- Revenue information
- Automotive sales performance
- Energy generation and storage metrics
- Operating expenses
- Risk factors
- Corporate governance details

The reports collectively contain several hundred pages of financial disclosures.


# Environment Setup

The following libraries are required:

- ChromaDB
- LangChain
- Sentence Transformers
- NVIDIA API SDK
- PyPDF Loader

The environment is configured before initializing the retrieval pipeline.

In [1]:
import time
import chromadb
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader

from datetime import datetime


c:\Agile\Agile_Assignments\AG0846_03\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["NVIDIA_API_KEY"] = os.getenv('NVIDIA_API_KEY')

In [3]:
# from groq import Groq

# client = Groq()

from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv('NVIDIA_API_KEY')
)

In [4]:
model_name = "meta/llama-3.1-70b-instruct"

 Instantiating the embedding model

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Ashish Sinha\AppData\Local\Temp\ipykernel_18228\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


# Load the Vector Database

Since we persisted the database to to a folder, we can upload this database to this Colab instance and point a Chroma instance to this database.

In practise, the database is maintained as a separate entity and CRUD operations are managed just as one would for normal databases (e.g., relational databases).

Now that the database is uploaded onto the Colab instance, we can unzip it and attach a retriever.

In [6]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [7]:
chromadb_client

In [11]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [12]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [13]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [14]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000145BD541EE0>, search_kwargs={'k': 5})

# RAG Q&A

## Prompt Design

The RAG system message should clearly communicate to the LLM that the input will include a user query along with the necessary context information to address that query. Additionally, the response should rely solely on the context information provided.

In [15]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [16]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

## Retrieving relevant documents

In [17]:
user_query = "What was the automotive revenue in 2021?"

In [18]:
relevant_document_chunks = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [19]:
len(relevant_document_chunks)

5

We can inspect the first document like so:

In [20]:
for document in relevant_document_chunks:
    print(document)
    break

page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

In [21]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
2022
2021
$
%
$
%
Automotive sales
$
78,509 
$
67,210 
$
44,125 
$
11,299 
17 
%
$
23,085 
52 
%
Automotive regulatory credits
1,790 
1,776 
1,465 
14 
1 
%
311 
21 
%
Automotive leasing
2,120 
2,476 
1,642 
(356)
(14)
%
834 
51 
%
Total automotive revenues
82,419 
71,462 
47,232 
10,957 
15 
%
24,230 
51 
%
Services and other
8,319 
6,091 
3,802 
2,228 
37 
%
2,289 
60 
%
Total automotive & services and other segment
revenue
90,738 
77,553 
51,034 
13,185 
17 
%
26,519 
52 
%
Energy generation and storage segment revenue
6,035 
3,909 
2,789 
2,126 
54 
%
1,120 
40 
%
Total revenues
$
96,773 
$
81,462 
$
53,823 
$
15,311 
19 
%
$
27,639 
51 
%
Automotive & Services and Other Segment
Automotive sales revenue includes revenues related to cash and financing deliveries of new Model S, Model X, Semi, Model 3, Model Y, and
Cybertruck vehicles, including access to our FSD Capability features and their ongoing maintenance, internet connectivity, free Supercharging program

## Composing the response

To compose the response to user queries, we assemble the prompt that uses the system message defined above and the dynamically retrieved context for the user query.

In [22]:
user_query = "What was the automotive revenue in 2021?"

In [23]:

# model_name = 'openai/gpt-oss-120b'
model_name = "meta/llama-3.1-70b-instruct"

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

$71,462


# Advanced Retrieval Strategy: Hypothetical Question Indexing

## Motivation

Standard semantic search may fail when a user query differs significantly from the wording used in the source documents.

To improve retrieval quality, we generate synthetic questions for every document chunk.

Instead of embedding only document text, we embed:

- Document chunks
- Generated hypothetical questions

This improves recall and enables more robust semantic matching.

# Hypothetical Questions

In this approach, we generate 3 hypothetical questions that can be answered with each document chunk. These hypothetical questions are then seperately indexed into the vector database (along with the parent document chunk ids as metadata). For each query, we then retrieve relevant hypothetical questions first and the then retrieve the associated chunks as the second step. Note that the retrieval is focused on the comparison between the user query and hypothetical questions.

# Lets Create Database for Hypothetical Questions

In [78]:
pdf_folder_location = "tesla-annual-reports"

In [79]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [80]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [81]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [82]:
collection = vectorstore_persisted._collection
total_docs = collection.count()
print(total_docs)

3337


In [83]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

Generating Hypothetical Questions: 0it [01:38, ?it/s]


In [84]:
len(tesla_10k_chunks)

3337

In [85]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

In [86]:
# print(tesla_10k_chunks[0].page_content)

# Database Creation

In [69]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [70]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [24]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [72]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'hypothetical_questions']

In [ ]:
# i = 0 # Initialize the starting index for the chunks

# while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
#     vectorstore.add_documents( # Add documents to the vector store in batches of 1500
#         documents=tesla_10k_chunks[i:i+500], # Get the current batch of 1500 chunks
#         ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
#     )

#     i += 500 # Increment the index by 500 to move to the next batch
#     time.sleep(5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [25]:
hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [ ]:
from tqdm.auto import tqdm
from langchain.schema import Document
import json
import time

hypothetical_questions = []
questions_json = []

batch_size = 10

successful_batches = 0
failed_batches = 0
skipped_batches = 0

# Adjust according to model context window
MAX_CHARS = 100000

total_batches = (
    len(tesla_10k_chunks) + batch_size - 1
) // batch_size

for batch_num, i in enumerate(
    tqdm(
        range(0, len(tesla_10k_chunks), batch_size),
        total=total_batches,
        desc="Generating Questions"
    ),
    start=1
):

    batch_docs = tesla_10k_chunks[i:i + batch_size]

    combined_text = "\n\n".join(
        doc.page_content
        for doc in batch_docs
    )

    char_count = len(combined_text)

    print(
        f"\nBatch {batch_num}/{total_batches}"
        f" | Chars={char_count:,}"
        f" | Docs={len(batch_docs)}"
    )

    # Skip oversized batches
    if char_count > MAX_CHARS:

        skipped_batches += 1

        print(
            f"Skipping batch {batch_num}"
            f" (too large: {char_count:,} chars)"
        )

        continue

    questions = ""

    wait_times = [5, 10, 15]

    for retry in range(len(wait_times)):

        try:

            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": hypothetical_questions_system_message
                    },
                    {
                        "role": "user",
                        "content": user_message_template.format(
                            document=combined_text
                        )
                    }
                ],
                temperature=0
            )

            questions = (
                response
                .choices[0]
                .message
                .content
                .strip()
            )

            successful_batches += 1

            print(
                f" Success"
                f" | Successful Batches: {successful_batches}"
            )

            break

        except Exception as e:

            error_msg = str(e).lower()

            print(
                f"\n Batch {batch_num}"
                f" | Retry {retry + 1}"
            )
            print(e)

            # Last retry
            if retry == len(wait_times) - 1:

                failed_batches += 1

                print(
                    f"Failed after "
                    f"{len(wait_times)} attempts"
                )

                questions = ""

                break

            wait_time = wait_times[retry]

            print(
                f"Waiting {wait_time}s "
                f"before retry..."
            )

            time.sleep(wait_time)

    if not questions:
        continue

    questions_list = [
        q.strip()
        for q in questions.split("\n")
        if q.strip()
    ]

    parent_doc = batch_docs[0]

    chunk_id = (
        parent_doc.metadata.get("chunk_id")
        or f"chunk_{i}"
    )

    for question in questions_list:

        metadata = {
            "parent_chunk_id": str(chunk_id),
            "parent_collection": "tesla_10k_collection"
        }

        # For Chroma
        hypothetical_questions.append(
            Document(
                page_content=question,
                metadata=metadata
            )
        )

        # For JSON
        questions_json.append({
            "question": question,
            "parent_chunk_id": str(chunk_id),
            "parent_collection": "tesla_10k_collection"
        })

    # Small delay between successful requests
    time.sleep(1)

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"Total Batches      : {total_batches}")
print(f"Successful Batches : {successful_batches}")
print(f"Failed Batches     : {failed_batches}")
print(f"Skipped Batches    : {skipped_batches}")
print(f"Questions Created  : {len(hypothetical_questions)}")

# Save JSON
with open(
    "tesla_hypothetical_questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        questions_json,
        f,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON saved successfully.")

Generating Questions:   0%|          | 0/334 [00:00<?, ?it/s]


Batch 1/334 | Chars=10,730 | Docs=10
 Success | Successful Batches: 1


Generating Questions:   0%|          | 1/334 [00:05<29:39,  5.34s/it]


Batch 2/334 | Chars=12,726 | Docs=10
 Success | Successful Batches: 2


Generating Questions:   1%|          | 2/334 [00:23<1:11:37, 12.94s/it]

...


Batch 333/334 | Chars=12,527 | Docs=10
 Success | Successful Batches: 325


Generating Questions: 100%|█████████▉| 333/334 [2:14:26<00:10, 10.79s/it]


Batch 334/334 | Chars=8,208 | Docs=7
 Success | Successful Batches: 326


Generating Questions: 100%|██████████| 334/334 [2:15:45<00:00, 24.39s/it]


SUMMARY
Total Batches      : 334
Successful Batches : 326
Failed Batches     : 8
Skipped Batches    : 0
Questions Created  : 978

JSON saved successfully.


In [91]:
import json

with open(
    "tesla_hypothetical_questions.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for record in questions_json:
        f.write(json.dumps(record) + "\n")

In [94]:
len(hypothetical_questions)

978

In [95]:
hypothetical_questions[0], hypothetical_questions[1], hypothetical_questions[2]

(Document(metadata={'parent_chunk_id': 'chunk_0', 'parent_collection': 'tesla_10k_collection'}, page_content="What is the composition of Tesla's Board of Directors as of April 28, 2022?"),
 Document(metadata={'parent_chunk_id': 'chunk_0', 'parent_collection': 'tesla_10k_collection'}, page_content='What is the background and qualifications of Elon Musk as the Chief Executive Officer of Tesla?'),
 Document(metadata={'parent_chunk_id': 'chunk_0', 'parent_collection': 'tesla_10k_collection'}, page_content='What is the aggregate market value of voting stock held by non-affiliates of Tesla as of June 30, 2021?'))

We can now index these hypothetical questions into a new collection.

In [96]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [97]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023', 'hypothetical_questions']

In [98]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_questions
)

['9489125d-706b-4d00-a35e-908b88c7875e',
 '07069ede-eb41-406b-9047-67f9b4526fa3',
 'd74f6e1c-5f64-44e3-99d4-b71ce9c899ed',
 '6b4b8875-5e96-4178-bf81-cb370099037b',
 'b5b40696-b0e1-4813-8c01-c7eb42a55752',
 '04242b20-5b59-4d2b-ad3c-9f06757b3400',
 '84ab2242-d881-48d0-9010-e909228516b8',
 '808f91c2-76a2-4e02-a5fa-988c7c07b5a4',
 '9fd4e291-51f8-4db6-85bf-bb059874ae03',
 '9d26aeca-81ce-45ea-bb49-320900902af5',
 '8cb2c50e-37c6-435a-a73e-509504a24e21',
 '5e9b5f58-2067-4d56-ab8d-fb606c7f32bf',
 'e0556041-c838-4dd9-abbf-42ee18d3e6df',
 '2cd10fe7-6a6a-4e9c-b61e-5318514121b4',
 '7fb8ee78-8656-4bbb-9d98-b4e074bf63e2',
 'e3416c12-7329-4ce6-9028-e6086a740993',
 '5fe666e2-6abc-419f-9a20-0bed77caa328',
 '4010d1c5-f0ad-4fbb-8bb5-f4c0a54793d8',
 '5e01bc9c-a36f-4ba6-81b5-a39d914530b7',
 '8363dca7-58fd-4369-9b78-072c71e14956',
 'b7880851-894c-4c7e-ae4d-8bf95c40fa05',
 '1ea0046d-88d5-46e9-a65e-c3ed443d460f',
 '4336b3a6-3b3f-489a-ba1d-64be72f6b845',
 '2ebb140c-cd5b-4055-b96d-39780953781f',
 '0efb7e12-9f76-

In [105]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [84]:
user_query = "What risks were highlighted in 2023?"

In [85]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

In [86]:
hypothetical_questions_retrieved

[Document(id='text_911', metadata={'creationdate': '2021-02-08T12:40:35+00:00', 'creator': 'wkhtmltopdf 0.12.5', 'page': 13, 'page_label': '14', 'producer': 'Qt 5.11.3', 'source': 'tesla-annual-reports\\tsla-10k_20201231-gen.pdf', 'title': '', 'total_pages': 449}, page_content='ITEM\t1A.\tRISK\tFACTORS\nYou\tshould\tcarefully\tconsider\tthe\trisks\tdescribed\tbelow\ttogether\twith\tthe\tother\tinformation\tset\tforth\tin\tthis\treport,\twhich\tcould\tmaterially\taffect\tour\nbusiness,\tfinancial\tcondition\tand\tfuture\tresults.\tThe\trisks\tdescribed\tbelow\tare\tnot\tthe\tonly\trisks\tfacing\tour\tcompany.\tRisks\tand\tuncertainties\tnot\tcurrently\nknown\tto\tus\tor\tthat\twe\tcurrently\tdeem\tto\tbe\timmaterial\talso\tmay\tmaterially\tadversely\taffect\tour\tbusiness,\tfinancial\tcondition\tand\toperating\tresults.\nRisks\tRelated\tto\tOur\tAbility\tto\tGrow\tOur\tBusiness\nWe\tmay\tbe\timpacted\tby\tmacroeconomic\tconditions\tresulting\tfrom\tthe\tglobal\tCOVID-19\tpandemic.\nSinc

In [87]:
for d in hypothetical_questions_retrieved:
    print(d)
    break

page_content='ITEM	1A.	RISK	FACTORS
You	should	carefully	consider	the	risks	described	below	together	with	the	other	information	set	forth	in	this	report,	which	could	materially	affect	our
business,	financial	condition	and	future	results.	The	risks	described	below	are	not	the	only	risks	facing	our	company.	Risks	and	uncertainties	not	currently
known	to	us	or	that	we	currently	deem	to	be	immaterial	also	may	materially	adversely	affect	our	business,	financial	condition	and	operating	results.
Risks	Related	to	Our	Ability	to	Grow	Our	Business
We	may	be	impacted	by	macroeconomic	conditions	resulting	from	the	global	COVID-19	pandemic.
Since	the	first	quarter	of	2020,	there	has	been	a	worldwide	impact	from	the	COVID-19	pandemic.	Government	regulations	and	shifting	social	behaviors
have	limited	or	closed	non-essential	transportation,	government	functions,	business	activities	and	person-to-person	interactions.	
In	some	cases,	the	relaxation	of
such	trends	has	recently	been	followed	by	actual	or	

In [88]:
retrieved_documents = vectorstore.get(
    ids=list(set([d.id for d in hypothetical_questions_retrieved]))
)

In [89]:
print(retrieved_documents)

{'ids': ['text_879', 'text_911', 'text_977', 'text_2233', 'text_2266'], 'embeddings': None, 'documents': ['Forward-Looking\tStatements\nThe\tdiscussions\tin\tthis\tAnnual\tReport\ton\tForm\t10-K\tcontain\tforward-looking\tstatements\treflecting\tour\tcurrent\texpectations\tthat\tinvolve\trisks\tand\tuncertainties.\nThese\tforward-looking\tstatements\tinclude,\tbut\tare\tnot\tlimited\tto,\tstatements\tconcerning\tany\tpotential\tfuture\timpact\tof\tthe\tcoronavirus\tdisease\t(“COVID-19”)\npandemic\ton\tour\tbusiness,\tour\tstrategy,\tfuture\toperations,\tfuture\tfinancial\tposition,\tfuture\trevenues,\tprojected\tcosts,\tprofitability,\texpected\tcost\treductions,\ncapital\tadequacy,\texpectations\tregarding\tdemand\tand\tacceptance\tfor\tour\ttechnologies,\tgrowth\topportunities\tand\ttrends\tin\tthe\tmarket\tin\twhich\twe\toperate,\nprospects\tand\tplans\tand\tobjectives\tof\tmanagement.\tThe\twords\t“anticipates,”\t“believes,”\t“could,”\t“estimates,”\t“expects,”\t“intends,”\t“may,”\t

In [90]:
docs = retriever.invoke(
    "What was Tesla's revenue in 2023?"
)

print(len(docs))

for doc in docs[:2]:
    print(doc.page_content[:500])

5
November	2021	for	additional	taxes	owed,	if	any.
	
Tesla	periodically	does	business	with	certain	entities	with	which	its	CEO	and	directors	are	affiliated,	such	as	SpaceX	and	Twitter,	Inc.,	in
	
accordance	with	our	Related	Person	Transactions	Policy.	Such	transactions	have	not	had	to	date,	and	are	not	currently	expected	to	have,	a	material
	
impact	on	our	consolidated	financial	statements.
	
Note	18	–	Segment	Reporting	and	Information	about	Geographic	Areas
We	have	
two
	operating	and	reportable	
objectives.	In	2021,	Tesla’s	full-year	accomplishments	under	our	executive	leadership	included	the	following:
	
•
Total	revenues	of	$53.82	billion,	representing	an	increase	of	$22.28	billion,	or	70.64%	compared	to	the	prior	year;
	
•
Net	income	attributable	to	common	stockholders	of	$5.52	billion	and	an	operating	margin	of	12.1%,	representing	favorable	changes	of
$4.80	billion	and	5.8%,	respectively,	compared	to	the	prior	year;
	
•
Annual	vehicle	delivery	and	production	records	of	936,222	and

In [91]:
retrieved_documents['documents']

['Forward-Looking\tStatements\nThe\tdiscussions\tin\tthis\tAnnual\tReport\ton\tForm\t10-K\tcontain\tforward-looking\tstatements\treflecting\tour\tcurrent\texpectations\tthat\tinvolve\trisks\tand\tuncertainties.\nThese\tforward-looking\tstatements\tinclude,\tbut\tare\tnot\tlimited\tto,\tstatements\tconcerning\tany\tpotential\tfuture\timpact\tof\tthe\tcoronavirus\tdisease\t(“COVID-19”)\npandemic\ton\tour\tbusiness,\tour\tstrategy,\tfuture\toperations,\tfuture\tfinancial\tposition,\tfuture\trevenues,\tprojected\tcosts,\tprofitability,\texpected\tcost\treductions,\ncapital\tadequacy,\texpectations\tregarding\tdemand\tand\tacceptance\tfor\tour\ttechnologies,\tgrowth\topportunities\tand\ttrends\tin\tthe\tmarket\tin\twhich\twe\toperate,\nprospects\tand\tplans\tand\tobjectives\tof\tmanagement.\tThe\twords\t“anticipates,”\t“believes,”\t“could,”\t“estimates,”\t“expects,”\t“intends,”\t“may,”\t“plans,”\t“projects,”\n“will,”\t“would”\tand\tsimilar\texpressions\tare\tintended\tto\tidentify\tforward-

"retrieved_documents" consists of those documents that hold answer to the user query. Hence they can be passed as retrived context to LLM to get final answer

In [92]:
len(retrieved_documents['documents'])

5

In [93]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [94]:
content_for_query = set(retrieved_documents)
content_for_query

{'data', 'documents', 'embeddings', 'ids', 'included', 'metadatas', 'uris'}

In [95]:
prompts = [
    {'role':'system','content':qna_system_message},
    {'role':'user','content':qna_user_message_template.format(
        context = content_for_query,
        question= user_query
    )}
]

In [96]:
try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompts,
        temperature=0.5
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

I don't know


# Conclusion

This project implemented a production-style Retrieval-Augmented Generation pipeline for Tesla annual reports.

Key achievements:

- Built a semantic search system using ChromaDB
- Integrated Llama 3.1 70B for grounded generation
- Implemented advanced retrieval using hypothetical questions
- Improved retrieval quality for complex financial queries

The approach demonstrates how RAG can be applied to enterprise knowledge bases and financial document analysis.

# Future Enhancements

Potential improvements include:

1. Hybrid Search
   - BM25 + Vector Search

2. Reranking Models
   - Cross Encoder Rerankers

3. Parent-Child Retrieval

4. Multi-Query Retrieval

5. Metadata Filtering
   - Year-based filtering
   - Report section filtering

6. Automated Evaluation Framework

7. Citation Generation
   - Return source pages with answers